In [2]:
import os
import pathlib

import numpy as np
import pandas as pd
import streamlit as st
from app_utils import latent_load_data, load_model_data, single_load_data
from sklearn.decomposition import PCA

2026-06-09 13:29:16.828 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-09 13:29:16.829 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-09 13:29:16.830 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-09 13:29:16.830 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-09 13:29:16.831 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Base directory: /home/lippincm/gene-process-dependencies/9.webapp
Repo root: /home/lippincm/gene-process-dependencies


In [3]:
_base_reactome, _base_corum, _base_drug = latent_load_data()
ALL_DISEASES = sorted(_base_reactome["OncotreePrimaryDisease"].unique().tolist())
ALL_MODEL_IDS = sorted(_base_reactome["ModelID"].unique().tolist())
# Go up one level to repo root, then into data/
BASE_DIR = pathlib.Path().resolve()
REPO_ROOT = BASE_DIR.parent
REPO_ROOT = BASE_DIR.parent  # goes up from 9.webapp/ to repo root
print(f"Base directory: {BASE_DIR}")
print(f"Repo root: {REPO_ROOT}")

2026-06-09 13:29:16.839 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-09 13:29:16.840 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Base directory: /home/lippincm/gene-process-dependencies/9.webapp
Repo root: /home/lippincm/gene-process-dependencies


In [46]:
combined_df = single_load_data()
data_directory = BASE_DIR / "data"
dependency_file = data_directory / "CRISPRGeneEffect.parquet"
gene_dict_file = data_directory / "CRISPR_gene_dictionary.parquet"
cancer_type_input_file = BASE_DIR / "data" / "Model.parquet"
dependency_df, gene_dict_df = load_model_data(dependency_file, gene_dict_file)
dependency_df = dependency_df.set_index("ModelID")
cancer_type_input_file = pathlib.Path(f"{BASE_DIR}/data/Model.parquet")
cancer_type_df = pd.read_parquet(cancer_type_input_file)
reactome_matrix, corum_matrix, drug_matrix = latent_load_data()

(1150, 18444)


In [63]:
corum_matrix_path = (
    REPO_ROOT / "5.drug-dependency" / "results" / "all_corum_results.parquet"
)
corum_matrix = pd.read_parquet(corum_matrix_path)
corum_matrix["feature"] = corum_matrix["reactome_pathway"]
corum_matrix = corum_matrix.drop(
    columns=["model", "latent_dim_total", "init", "seed", "z", "reactome_pathway"]
)
meta_cols = ["ModelID", "OncotreePrimaryDisease"]

corum_meta = corum_matrix[meta_cols].drop_duplicates()

# corum_matrix = corum_matrix.pivot(index="ModelID", columns="feature", values="latent_score")
# corum_matrix = corum_matrix.merge(corum_meta, on="ModelID", how="left")
# print(corum_matrix.isna().sum().sum())
# corum_matrix.drop(columns=['gsea_es_score',"pathway_score","OncotreePrimaryDisease"], inplace=True)
# corum_matrix.pivot(index="ModelID", columns="feature", values="latent_score")
corum_matrix

,ModelID,OncotreePrimaryDisease,latent_score,gsea_es_score,pathway_score,feature
0,ACH-000001,Ovarian Epithelial Tumor,0.392860,0.824315,0.392860,"Inactivation, Recovery And Regulation Of Photo..."
1,ACH-000001,Ovarian Epithelial Tumor,0.763630,0.880116,0.763630,Elastic Fibre Formation R-HSA-1566948
2,ACH-000001,Ovarian Epithelial Tumor,0.435531,0.839506,0.435531,Mismatch Repair (MMR) Directed By MSH2:MSH6 (M...
3,ACH-000001,Ovarian Epithelial Tumor,0.531491,-0.733375,0.531491,Processive Synthesis On Lagging Strand R-HSA-6...
4,ACH-000001,Ovarian Epithelial Tumor,0.267750,0.864257,0.267750,Mismatch Repair (MMR) Directed By MSH2:MSH3 (M...
...,...,...,...,...,...,...
169777,ACH-003012,Non-Small Cell Lung Cancer,0.505296,0.938271,0.505296,Kinase maturation complex 1 (human)
169778,ACH-003012,Non-Small Cell Lung Cancer,0.282634,0.808217,0.282634,immunoproteasome (mouse)
169779,ACH-003012,Non-Small Cell Lung Cancer,0.073697,0.748024,0.073697,ALL-1 supercomplex (human)
169780,ACH-003012,Non-Small Cell Lung Cancer,0.032945,0.739678,0.032945,Emerin complex 32 (human)


In [61]:
for x in corum_matrix["feature"].unique():
    print(x)

Inactivation, Recovery And Regulation Of Phototransduction Cascade R-HSA-2514859
Elastic Fibre Formation R-HSA-1566948
Mismatch Repair (MMR) Directed By MSH2:MSH6 (MutSalpha) R-HSA-5358565
Processive Synthesis On Lagging Strand R-HSA-69183
Mismatch Repair (MMR) Directed By MSH2:MSH3 (MutSbeta) R-HSA-5358606
Phototransduction Cascade R-HSA-2514856
RNA polymerase II complex, chromatin structure modifying (human)
Integrator-RNAPII complex (human)
Parvulin-associated pre-rRNP complex (mouse)
PCAF complex (human)
DNA synthesome complex (17 subunits) (human)
CTF18-cohesion-RFC-POLH complex (human)
FCP1-associated protein complex (human)
CF IIAm complex (Cleavage factor IIAm complex) (human)
BRM-SIN3A-HDAC complex (human)
TNF-alpha/NF-kappa B signaling complex 6 (human)
TNF-alpha/NF-kappa B signaling complex (CHUK, BTRC, NFKB2, PPP6C, REL, CUL1, IKBKE, SAPS2, SAPS1, ANKRD28, RELA, SKP1) (human)
E2F-6 complex (human)
BRD4 complex (human)
RC complex (Replication competent complex) (human)
PBAF 

In [ ]:
# get all modelids that have NA values in the corum matrix
corum_matrix[corum_matrix.isna().any(axis=1)]["ModelID"].sort_values().unique()

In [18]:
# @st.cache_data


def compute_pca(df, metadata_columns) -> pd.DataFrame:
    """Cache PCA result per disease selection."""
    # df.fillna(0, inplace=True)  # Fill NA with 0 for PCA, or drop rows with NA
    print(df.shape[0], "rows before dropping NA")
    df.dropna(inplace=True)  # PCA can't handle missing values, so drop rows with any NA
    print(df.shape[0], "rows after dropping NA")
    metadata_df = df[metadata_columns]
    feat_cols = df.columns.drop(metadata_columns)
    if "source" in feat_cols:
        feat_cols = feat_cols.drop("source")

    pca_input = df[feat_cols].apply(pd.to_numeric, errors="coerce")
    pca = PCA(n_components=2, random_state=0)
    pca_embedding = pca.fit_transform(pca_input)
    metadata_df["PCA1"] = pca_embedding[:, 0]
    metadata_df["PCA2"] = pca_embedding[:, 1]
    return metadata_df


pca_single_df = compute_pca(combined_df, ["ModelID", "OncotreePrimaryDisease"])
# save the PCA result to a parquet file for later use in the app
pca_single_df.to_parquet(
    BASE_DIR / "data" / "pca_embeddings_single_dependencies.parquet"
)

pca_latent_reactome_df = compute_pca(
    reactome_matrix, ["OncotreePrimaryDisease", "ModelID"]
)
pca_latent_corum_df = compute_pca(corum_matrix, ["OncotreePrimaryDisease", "ModelID"])
pca_latent_drug_df = compute_pca(drug_matrix, ["OncotreePrimaryDisease", "ModelID"])
pca_latent_reactome_df.to_parquet(
    BASE_DIR / "data" / "pca_embeddings_latent_reactome.parquet"
)
pca_latent_corum_df.to_parquet(
    BASE_DIR / "data" / "pca_embeddings_latent_corum.parquet"
)
pca_latent_drug_df.to_parquet(BASE_DIR / "data" / "pca_embeddings_latent_drug.parquet")

1150 rows before dropping NA
1150 rows after dropping NA
1150 rows before dropping NA
958 rows after dropping NA
1150 rows before dropping NA
958 rows after dropping NA
1150 rows before dropping NA
958 rows after dropping NA
